# Chapter 11 Training Deep Neural Networks
### 11.1 The Vanishing/Exploding Gradients Problems

In [1]:
import tensorflow as tf

physical_devices = tf.config.list_physical_devices("GPU")
for device in physical_devices:
    tf.config.experimental.set_memory_growth(device, True)

### Fetch data

In [2]:
import pandas as pd
from sklearn.datasets import fetch_openml

mnist = fetch_openml("mnist_784", version=1)
X, y = mnist["data"], mnist["target"]

### Explore data

In [3]:
X.head()

,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,pixel10,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [4]:
y.head()

0    5
1    0
2    4
3    1
4    9
Name: class, dtype: category
Categories (10, object): ['0', '1', '2', '3', ..., '6', '7', '8', '9']

In [5]:
X = X.values
y = y.values

In [6]:
X[:5]

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

In [7]:
y[:5]

['5', '0', '4', '1', '9']
Categories (10, object): ['0', '1', '2', '3', ..., '6', '7', '8', '9']

In [8]:
X.shape

(70000, 784)

In [9]:
y.shape

(70000,)

__visualize data__

skipped

### Prepare data

__Normalize data__

In [10]:
X_norm = X / 255

In [11]:
from sklearn.preprocessing import LabelBinarizer

lb = LabelBinarizer()
y_onehot = lb.fit_transform(y)

__Split data__

In [12]:
from sklearn.model_selection import train_test_split

X_train_valid, X_test, y_train_valid, y_test = train_test_split(
    X_norm, y_onehot, test_size=0.2, random_state=42
)

In [13]:
X_train_valid.shape

(56000, 784)

In [14]:
X_test.shape

(14000, 784)

In [15]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_valid, y_train_valid, test_size=0.2, random_state=42
)

In [16]:
X_train.shape

(44800, 784)

In [17]:
X_valid.shape

(11200, 784)

### Shortlist promising models

In [18]:
from tensorflow import keras


def create_model(activation="relu", kernel_initializer="glorot_uniform"):
    model = keras.Sequential(
        [
            keras.layers.InputLayer(input_shape=(28 * 28)),
            keras.layers.BatchNormalization(),
            keras.layers.Dense(
                300,
                activation=activation,
                kernel_initializer=kernel_initializer,
                use_bias=False,
            ),
            keras.layers.BatchNormalization(),
            keras.layers.Dense(
                100,
                activation=activation,
                kernel_initializer=kernel_initializer,
                use_bias=False,
            ),
            keras.layers.BatchNormalization(),
            keras.layers.Dense(10, activation="softmax"),
        ]
    )
    model.compile(optimizer="adam", loss="categorical_crossentropy", metrics="acc")
    return model


models = [
    create_model(),
    create_model("elu", "he_normal"),
    create_model("selu", kernel_initializer="lecun_normal"),
]

models[0].summary()

Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
batch_normalization (BatchNo (None, 784)               3136      
_________________________________________________________________
dense (Dense)                (None, 300)               235200    
_________________________________________________________________
batch_normalization_1 (Batch (None, 300)               1200      
_________________________________________________________________
dense_1 (Dense)              (None, 100)               30000     
_________________________________________________________________
batch_normalization_2 (Batch (None, 100)               400       
_________________________________________________________________
dense_2 (Dense)              (None, 10)                1010      
Total params: 270,946
Trainable params: 268,578
Non-trainable params: 2,368
______________________________________________

In [19]:
import numpy as np
from sklearn.model_selection import KFold

n_models = len(models)
n_splits = 5
scores = np.zeros((n_splits, n_models))

kf = KFold(n_splits=n_splits)
for i_split, (ids_train, ids_valid) in enumerate(
    kf.split(X_train_valid, y_train_valid)
):
    for i_model, model in enumerate(models):
        history = model.fit(
            X_train_valid[ids_train],
            y_train_valid[ids_train],
            epochs=1000,
            callbacks=[keras.callbacks.EarlyStopping(patience=10)],
            validation_data=(X_train_valid[ids_valid], y_train_valid[ids_valid]),
        )
        scores[i_split, i_model] = model.evaluate(
            X_train_valid[ids_valid], y_train_valid[ids_valid]
        )[1]

df_scores = pd.DataFrame(data=scores, columns=["relu", "elu", "selu"])

Epoch 1/1000
1400/1400 [==============================] - 5s 3ms/step - loss: 0.3776 - acc: 0.8853 - val_loss: 0.1348 - val_acc: 0.9611
Epoch 2/1000
1400/1400 [==============================] - 4s 3ms/step - loss: 0.1177 - acc: 0.9623 - val_loss: 0.1064 - val_acc: 0.9696
Epoch 3/1000
1400/1400 [==============================] - 4s 3ms/step - loss: 0.0839 - acc: 0.9746 - val_loss: 0.1025 - val_acc: 0.9697
Epoch 4/1000
1400/1400 [==============================] - 3s 2ms/step - loss: 0.0644 - acc: 0.9801 - val_loss: 0.1092 - val_acc: 0.9708
Epoch 5/1000
1400/1400 [==============================] - 4s 3ms/step - loss: 0.0542 - acc: 0.9824 - val_loss: 0.1080 - val_acc: 0.9721
Epoch 6/1000
1400/1400 [==============================] - 4s 3ms/step - loss: 0.0452 - acc: 0.9854 - val_loss: 0.0874 - val_acc: 0.9767
Epoch 7/1000
1400/1400 [==============================] - 3s 2ms/step - loss: 0.0375 - acc: 0.9873 - val_loss: 0.0958 - val_acc: 0.9754
Epoch 8/1000
1400/1400 [========================

In [20]:
df_scores

,relu,elu,selu
0,0.976429,0.975357,0.973571
1,0.989375,0.992679,0.989643
2,0.994911,0.993839,0.994911
3,0.995893,0.996071,0.995714
4,0.995982,0.997500,0.995714


In [21]:
df_scores_mean_std = pd.DataFrame({"mean": df_scores.mean(), "std": df_scores.std()})
df_scores_mean_std

,mean,std
relu,0.990518,0.008335
elu,0.991089,0.008993
selu,0.989911,0.009479


__Train (the best) model__

In [22]:
model = create_model()
model.fit(
    X_train,
    y_train,
    epochs=1000,
    callbacks=[keras.callbacks.EarlyStopping(patience=10)],
    validation_data=(X_valid, y_valid),
)

Epoch 1/1000
1400/1400 [==============================] - 4s 3ms/step - loss: 0.3840 - acc: 0.8819 - val_loss: 0.1379 - val_acc: 0.9593
Epoch 2/1000
1400/1400 [==============================] - 3s 2ms/step - loss: 0.1181 - acc: 0.9634 - val_loss: 0.1044 - val_acc: 0.9681
Epoch 3/1000
1400/1400 [==============================] - 4s 3ms/step - loss: 0.0879 - acc: 0.9723 - val_loss: 0.1024 - val_acc: 0.9693
Epoch 4/1000
1400/1400 [==============================] - 4s 3ms/step - loss: 0.0672 - acc: 0.9778 - val_loss: 0.0911 - val_acc: 0.9727
Epoch 5/1000
1400/1400 [==============================] - 3s 2ms/step - loss: 0.0519 - acc: 0.9826 - val_loss: 0.0952 - val_acc: 0.9729
Epoch 6/1000
1400/1400 [==============================] - 4s 3ms/step - loss: 0.0451 - acc: 0.9853 - val_loss: 0.0891 - val_acc: 0.9745
Epoch 7/1000
1400/1400 [==============================] - 4s 3ms/step - loss: 0.0427 - acc: 0.9855 - val_loss: 0.0966 - val_acc: 0.9728
Epoch 8/1000
1400/1400 [========================

__Evaluate the model__

In [23]:
loss, acc = model.evaluate(X_test, y_test)

438/438 [==============================] - 1s 2ms/step - loss: 0.1205 - acc: 0.9754


__Use the model for prediction__

In [24]:
X_new = X_test[:5]
y_new = y_test[:5]

y_pred = model.predict(X_new)

In [25]:
np.argmax(y_new, axis=1)

array([8, 4, 8, 7, 7])

In [26]:
np.argmax(y_pred, axis=1)

array([8, 4, 8, 7, 7])

### Examine BN layer

In [27]:
[(var.name, var.trainable) for var in model.layers[0].variables]

[('batch_normalization_9/gamma:0', True),
 ('batch_normalization_9/beta:0', True),
 ('batch_normalization_9/moving_mean:0', False),
 ('batch_normalization_9/moving_variance:0', False)]

In [28]:
model.layers[0].updates

/home/yu/anaconda3/envs/data-science/lib/python3.9/site-packages/tensorflow/python/keras/engine/base_layer.py:1402: UserWarning: `layer.updates` will be removed in a future version. This property should not be used in TensorFlow 2.0, as `updates` are applied automatically.
  warnings.warn('`layer.updates` will be removed in a future version. '


[]